# Exploration du dataset GDIZ

50 machines textiles simulées sur deux ans (2024-2025), un relevé toutes les 30 minutes, avec la météo réelle de
Glo-Djigbé (Open-Meteo). Les données sont générées par `make generate-data` dans `data/raw/gdiz_maintenance.db`.

In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect("../data/raw/gdiz_maintenance.db", read_only=True)
conn.sql("SHOW TABLES").df()

,name
0,factory_telemetry
1,meteo_exterieure
2,predictions


## Volume et période

In [2]:
conn.sql("""SELECT COUNT(*) AS releves, COUNT(DISTINCT id_machine) AS machines,
                  MIN(timestamp) AS debut, MAX(timestamp) AS fin FROM factory_telemetry""").df()

,releves,machines,debut,fin
0,1752000,50,2024-01-01,2025-12-30 23:30:00


## États des machines

`marche` : en production ; `arret` : arrêt planifié, hors production ; `delestage` : coupure du réseau SBEE, la machine
ne tourne pas et ne mesure rien ; `panne` : réparation après une panne ; `entretien` : visite préventive.

In [3]:
conn.sql("""SELECT etat, COUNT(*) AS releves, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS part_pct
            FROM factory_telemetry GROUP BY etat ORDER BY releves DESC""").df()

,etat,releves,part_pct
0,marche,1243143,70.96
1,arret,496292,28.33
2,delestage,7152,0.41
3,panne,5017,0.29
4,entretien,396,0.02


## Pannes par gamme et par mode

TWF usure, HDF surchauffe, OSF surcharge mécanique : précédées de signes avant-coureurs, ce sont les pannes que le
modèle apprend. PWF (électrique) et RNF (aléatoire) n'en ont pas.

In [4]:
conn.sql("""PIVOT (SELECT type_machine AS gamme, mode_panne FROM factory_telemetry WHERE panne = 1)
            ON mode_panne USING COUNT(*) ORDER BY gamme""").df()

,gamme,HDF,OSF,PWF,RNF,TWF
0,H,7,18,6,6,22
1,L,55,26,47,8,46
2,M,8,15,37,6,29


## Capteurs : valeurs typiques et valeurs manquantes

In [5]:
capteurs = ["temperature_process", "vibration", "couple", "concentration_poussiere", "niveau_tension"]
df = conn.sql(f"SELECT {', '.join(capteurs)} FROM factory_telemetry WHERE etat = 'marche'").df()
pd.DataFrame({"mediane": df.median(), "p99": df.quantile(0.99), "manquants_pct": df.isna().mean() * 100}).round(2)

,mediane,p99,manquants_pct
temperature_process,68.03,77.58,1.00
vibration,1.37,4.05,1.00
couple,42.72,55.19,0.94
concentration_poussiere,39.13,252.86,1.01
niveau_tension,208.29,216.90,0.00


In [6]:
conn.close()